In [1]:
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
import matplotlib.pyplot as plt
import pickle

In [2]:
from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession
import os
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.9
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from load_network import load_VGG
from train import WarmUpCosine, CustomWeightDecaySGD
from save_load import load_noise_if_exists, save_layer_outputs_and_labels, load_layer_outputs_and_labels
from weights_bias_cla import load_wb_if_exists
from evaluate_cla import evalu_stream_main_selected, per_group_ovr_accuracy, evalu_select

In [4]:
with open('data.pkl', 'rb') as f:
    [x_train,y_train,x_test,y_test]=pickle.load(f)
y_train_onehot=tf.keras.utils.to_categorical(y_train,num_classes=4)
y_test_onehot=tf.keras.utils.to_categorical(y_test,num_classes=4)

In [5]:
model=load_VGG()

In [6]:
pred_model = tf.keras.Model(
    inputs=model.get_layer("re_lu_9").output,
    outputs=model.output
)

In [7]:
l_label = [2,6,10,13,17,20,24,27,32,35]

In [8]:
layer_list = [model.layers[l].name for l in l_label]

In [9]:
NOISE_DIR = "./noise/"
save_layer_outputs_and_labels(model, x_train, y_train, layer_list, save_dir="D:/Data_3/VGG-11/layer_cache/base")
for i in range(10):
    NOISE_I_DIR = NOISE_DIR + str(i)
    x_gauss, x_salt, x_move, x_occ = load_noise_if_exists(NOISE_I_DIR)
    save_layer_outputs_and_labels(model, x_gauss, y_train, layer_list, save_dir="D:/Data_3/VGG-11/layer_cache/gauss/"+str(i))
    save_layer_outputs_and_labels(model, x_salt, y_train, layer_list, save_dir="D:/Data_3/VGG-11/layer_cache/salt/"+str(i))

[SAVE] Generating layer outputs for: D:/Data_3/VGG-11/layer_cache/base
[Saved] re_lu: outputs (20000, 32768), labels (20000, 1)
[Saved] re_lu_1: outputs (20000, 8192), labels (20000, 1)
[Saved] re_lu_2: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_3: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_4: outputs (20000, 2048), labels (20000, 1)
[Saved] re_lu_5: outputs (20000, 2048), labels (20000, 1)
[Saved] re_lu_6: outputs (20000, 1024), labels (20000, 1)
[Saved] re_lu_7: outputs (20000, 1024), labels (20000, 1)
[Saved] re_lu_8: outputs (20000, 256), labels (20000, 1)
[Saved] re_lu_9: outputs (20000, 256), labels (20000, 1)
[SAVE] Generating layer outputs for: D:/Data_3/VGG-11/layer_cache/gauss/0
[Saved] re_lu: outputs (20000, 32768), labels (20000, 1)
[Saved] re_lu_1: outputs (20000, 8192), labels (20000, 1)
[Saved] re_lu_2: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_3: outputs (20000, 4096), labels (20000, 1)
[Saved] re_lu_4: outputs (20000, 2048), labels 

In [10]:
CACHE_DIR = "./VGG-11/w_and_b_cache"

In [11]:
eva_w,eva_b = load_wb_if_exists(y_train, layer_list, CACHE_DIR, save_dir="D:/Data_3/VGG-11/layer_cache/base")


==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 13971
xi>=0 count: 14306
xi>=0 count: 12570
xi>=0 count: 14332

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 15306
xi>=0 count: 14682
xi>=0 count: 14457
xi>=0 count: 14138

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 15877
xi>=0 count: 15389
xi>=0 count: 14721
xi>=0 count: 13982

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 16746
xi>=0 count: 16462
xi>=0 count: 16011
xi>=0 count: 15978

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 16927
xi>=0 count: 17767
xi>=0 count: 17737
xi>=0 count: 16892

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count: 17657
xi>=0 count: 17991
xi>=0 count: 18063
xi>=0 count: 16527

==== split 0 ====

==== split 1 ====

==== split 2 ====

==== split 3 ====
xi>=0 count:

In [12]:
NOISE_dir='./noise/'
ATTACK_dir=Noise_dir='./attack/'

In [13]:
select_list = evalu_select(layer_list, y_train, eva_w, eva_b, pred_model=pred_model, save_dir = "D:/Data_3/VGG-11/layer_cache/base")

[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]
[3 0 2 ... 3 2 2]


In [14]:
base = evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_3/VGG-11/layer_cache/base")
base_final = per_group_ovr_accuracy(model, x_train, y_train, select_list)
base = np.concatenate((base,base_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.64769474 0.63523091 0.68622771 0.62983076]
Layer 1
accuracy: [0.72154001 0.70192898 0.75017785 0.67140855]
Layer 2
accuracy: [0.72635775 0.75871033 0.76649497 0.64736619]
Layer 3
accuracy: [0.77871323 0.7905094  0.78695769 0.66817418]
Layer 4
accuracy: [0.77135607 0.87873786 0.8519646  0.74357896]
Layer 5
accuracy: [0.80463778 0.89172224 0.89202527 0.73143876]
Layer 6
accuracy: [0.94124682 0.96151451 0.96121346 0.9524884 ]
Layer 7
accuracy: [0.96351701 0.95506923 0.96198661 0.93432074]
Layer 8
accuracy: [0.98550185 0.98851184 0.992266   0.9822294 ]
Layer 9
accuracy: [0.94545214 0.95373229 0.96972093 0.87337703]


In [15]:
base

array([[0.64769474, 0.63523091, 0.68622771, 0.62983076],
       [0.72154001, 0.70192898, 0.75017785, 0.67140855],
       [0.72635775, 0.75871033, 0.76649497, 0.64736619],
       [0.77871323, 0.7905094 , 0.78695769, 0.66817418],
       [0.77135607, 0.87873786, 0.8519646 , 0.74357896],
       [0.80463778, 0.89172224, 0.89202527, 0.73143876],
       [0.94124682, 0.96151451, 0.96121346, 0.9524884 ],
       [0.96351701, 0.95506923, 0.96198661, 0.93432074],
       [0.98550185, 0.98851184, 0.992266  , 0.9822294 ],
       [0.94545214, 0.95373229, 0.96972093, 0.87337703],
       [1.        , 1.        , 1.        , 1.        ]])

In [16]:
gauss = np.zeros((len(layer_list),4))
for i in range(10):
    gauss += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_3/VGG-11/layer_cache/gauss/"+str(i))
gauss = gauss/10
gauss_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/gauss.npy'
    x_noise = np.load(DIR)
    gauss_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
gauss_final = gauss_final/10
gauss = np.concatenate((gauss,gauss_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.54846653 0.34333852 0.55186302 0.3277637 ]
Layer 1
accuracy: [0.74955299 0.74838173 0.7498202  0.7359056 ]
Layer 2
accuracy: [0.74828662 0.76150006 0.75679349 0.72966232]
Layer 3
accuracy: [0.766569   0.77506942 0.76201548 0.74295474]
Layer 4
accuracy: [0.76030315 0.8049562  0.79271861 0.75672834]
Layer 5
accuracy: [0.7935636  0.8511076  0.83774667 0.75094889]
Layer 6
accuracy: [0.90286424 0.94219231 0.92155814 0.89601818]
Layer 7
accuracy: [0.90066868 0.9294032  0.93476465 0.90282395]
Layer 8
accuracy: [0.8957205  0.93503974 0.9267698  0.91197961]
Layer 9
accuracy: [0.88389632 0.9140698  0.94060095 0.87650304]
Layer 0
accuracy: [0.54372958 0.34089805 0.54855122 0.32401158]
Layer 1
accuracy: [0.7495486  0.74860006 0.74956793 0.73741609]
Layer 2
accuracy: [0.74679221 0.75969323 0.75778319 0.73021485]
Layer 3
accuracy: [0.7665429  0.7719223  0.76734244 0.74431237]
Layer 4
accuracy: [0.76209046 0.80378081 0.79355836 0.75901758]
Layer 5
accuracy: [0.79485244 0.84667411

In [17]:
gauss

array([[0.54492174, 0.34127318, 0.54885053, 0.32599457],
       [0.74955147, 0.74837988, 0.74964251, 0.73619968],
       [0.74775739, 0.76101006, 0.75627312, 0.73091975],
       [0.76642828, 0.77333281, 0.76515159, 0.7425501 ],
       [0.76275356, 0.80347068, 0.79331086, 0.75466406],
       [0.79589245, 0.84937589, 0.83514156, 0.74995392],
       [0.90828929, 0.94389348, 0.92196777, 0.8944738 ],
       [0.90767603, 0.93220394, 0.93404728, 0.89920735],
       [0.90054975, 0.93927511, 0.9274193 , 0.91229458],
       [0.8861806 , 0.91334979, 0.94286193, 0.87569201],
       [0.9513    , 0.9677    , 0.95495   , 0.951225  ]])

In [18]:
salt = np.zeros((len(layer_list),4))
for i in range(10):
    salt += evalu_stream_main_selected(layer_list, y_train, eva_w, eva_b, select_list, save_dir = "D:/Data_3/VGG-11/layer_cache/salt/"+str(i))
salt = salt/10
salt_final = np.zeros(4)
for i in range(10):
    DIR = NOISE_dir + str(i) + '/salt.npy'
    x_noise = np.load(DIR)
    salt_final += per_group_ovr_accuracy(model, x_noise, y_train, select_list)
salt_final = salt_final/10
salt = np.concatenate((salt,salt_final.reshape(1,4)),axis=0)

Layer 0
accuracy: [0.51416619 0.33556555 0.50087758 0.33080463]
Layer 1
accuracy: [0.74754491 0.74359021 0.74979828 0.72192051]
Layer 2
accuracy: [0.7418144  0.7769729  0.76561001 0.69833517]
Layer 3
accuracy: [0.78320778 0.79802747 0.79541271 0.72398814]
Layer 4
accuracy: [0.77428807 0.85447005 0.83297073 0.76542334]
Layer 5
accuracy: [0.81455609 0.87993628 0.87238846 0.75628085]
Layer 6
accuracy: [0.92391848 0.95906403 0.95279767 0.93024334]
Layer 7
accuracy: [0.93716157 0.95053651 0.9542356  0.9235082 ]
Layer 8
accuracy: [0.95016686 0.97448552 0.9692663  0.9579455 ]
Layer 9
accuracy: [0.92511815 0.94547808 0.96476673 0.8855921 ]
Layer 0
accuracy: [0.51369253 0.33693699 0.50417951 0.32753484]
Layer 1
accuracy: [0.74755928 0.7452769  0.75031713 0.72215347]
Layer 2
accuracy: [0.74211901 0.77783268 0.76409545 0.70234447]
Layer 3
accuracy: [0.78074509 0.79848857 0.78974841 0.72195893]
Layer 4
accuracy: [0.77333913 0.85745646 0.83550146 0.77565167]
Layer 5
accuracy: [0.81234133 0.88313501

In [19]:
salt

array([[0.51523571, 0.3371689 , 0.50393123, 0.32897549],
       [0.74783542, 0.74435554, 0.74970809, 0.72155591],
       [0.74242841, 0.77432144, 0.7651022 , 0.70311687],
       [0.78290084, 0.79799642, 0.79094539, 0.72323722],
       [0.77351339, 0.85474428, 0.83269785, 0.77087966],
       [0.80944102, 0.88263153, 0.87061629, 0.75747632],
       [0.92499813, 0.95715778, 0.94938135, 0.93096985],
       [0.93912284, 0.94798221, 0.95513602, 0.92225458],
       [0.94956196, 0.97025271, 0.96599027, 0.9538257 ],
       [0.92424446, 0.9406435 , 0.96161158, 0.88546186],
       [0.980925  , 0.98882499, 0.98139999, 0.98095   ]])

In [20]:
SAVE_FILE='e-VGG-11.pkl'

In [21]:
progress = {"base": base, "gauss": gauss, "salt": salt}
with open(SAVE_FILE, "wb") as f:
    pickle.dump(progress, f)

In [22]:
def stats_minmax_from_matrix_dict(matrix_dict, check_num=3): 
    """ 
    Input: 
        matrix_dict: {name: np.ndarray(m, n)}, a dictionary containing 3 matrices 
    Output: 
        stats: { 
            name: { 
                "mean": (m,), 
                "min":  (m,), 
                "max":  (m,) 
            }, ... 
        } 
    For each component i, statistics are computed along axis=1 (over n samples): 
        mean[i] = mean(X[i, :]) 
        min[i]  = min(X[i, :]) 
        max[i]  = max(X[i, :]) 
    """ 
    if check_num is not None and len(matrix_dict) != check_num: 
        raise ValueError(f"Expected {check_num} matrices, but received {len(matrix_dict)}") 
 
    # shape consistency 
    shapes = [np.asarray(v).shape for v in matrix_dict.values()] 
    if len(set(shapes)) != 1: 
        raise ValueError(f"Inconsistent matrix shapes: {shapes}") 
 
    stats = {} 
    for name, X in matrix_dict.items(): 
        X = np.asarray(X) 
        stats[name] = { 
            "mean": X.mean(axis=1), 
            "std":  X.std(axis=1), 
            "min":  X.min(axis=1), 
            "max":  X.max(axis=1), 
        } 
    return stats 

In [24]:
mean_var = stats_minmax_from_matrix_dict(progress)

ValueError: 期望 6 个矩阵，但收到 3 个

In [ ]:
mean_var